# 1、字符串解析器 StrOutputParser

In [5]:
from langchain_core.output_parsers import StrOutputParser
# 1.获取大模型
from langchain_openai import ChatOpenAI
import os
import dotenv

dotenv.load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
os.environ["OPENAI_BASE_URL"] = os.getenv("OPENAI_BASE_URL")

chat_model = ChatOpenAI(
    model = "gpt-4o-mini"
)
# 2.调用大模型
response = chat_model.invoke("什么是openclaw?")
print(type(response)) # AIMessage

# 3.如何获取一个字符串的输出结果
# 方式一：自己调用输出结果的content
#print(response.content)
print(type(response.content)) # str
# 方式二：使用StrOutputParser
parser = StrOutputParser()
str_response = parser.invoke(response)
print(str_response)
print(type(str_response))

<class 'langchain_core.messages.ai.AIMessage'>
<class 'str'>
OpenClaw 是一个开源项目，旨在为游戏开发提供一种可以方便地实现物理模拟和碰撞检测的工具。它通常被用在2D游戏中，以实现物体之间的交互和物理效果。OpenClaw 的设计目标是简化物理引擎的使用，使得开发人员能更容易地集成到他们的游戏项目中。

OpenClaw 的特点可能包括：

1. **开源和可扩展性**：由于是开源项目，开发者可以根据自己的需求修改和扩展代码。
2. **简易集成**：支持不同的游戏引擎或框架，减少了集成的复杂性。
3. **物理和碰撞检测**：提供基本的物理引擎功能，如重力、速度、碰撞检测等。

请注意，具体的功能和特性可能会随着项目的发展而变化。如果您有更多具体的需求或问题，欢迎进一步询问！
<class 'langchain_core.messages.base.TextAccessor'>


#  2、JSON解析器 JsonOutputParser

方法1：用户自己通过提示词指明返回Json格式

In [6]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

chat_model = ChatOpenAI(model="gpt-4o-mini")
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个靠谱的{role}"),
    ("human","{question}")
])
chat_prompt = chat_prompt_template.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说？问题用Q表示，答案用A表示，返回一个Json的数据。"})

response = chat_model.invoke(chat_prompt)
# 创建JsonOutputParser实例
parser = JsonOutputParser()
json_response = parser.invoke(response)
print(json_response)

{'Q': '人工智能用英文怎么说？', 'A': 'Artificial Intelligence'}


方法2：使用JsonOutputParser的get_format_instructions()

In [7]:
 #引入依赖包
from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts import PromptTemplate
# 初始化语言模型
chat_model = ChatOpenAI(model="gpt-4o-mini")
joke_query = "告诉我一个笑话。"
# 定义Json解析器
parser = JsonOutputParser()

prompt_template = PromptTemplate.from_template(
    template = "回答用户的查询\n 满足的格式为{format_instructions}\n 问题是{question}\n",
    partial_variables = {"format_instructions":parser.get_format_instructions()}
)

prompt = prompt_template.invoke(input = {"question":joke_query})
response = chat_model.invoke(prompt)
json_response = parser.invoke(response)
print(json_response)

{'joke': '为什么程序员喜欢在海边工作？因为那里有很多海洋（是）！'}


知识拓展：使用链式结构

In [8]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser

chat_model = ChatOpenAI(model="gpt-4o-mini")
chat_prompt_template = ChatPromptTemplate.from_messages([
    ("system","你是一个靠谱的{role}"),
    ("human","{question}")
])
# 创建JsonOutputParser实例
parser = JsonOutputParser()

# 写法一
chat_prompt = chat_prompt_template.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说？问题用Q表示，答案用A表示，返回一个Json的数据。"})

response = chat_model.invoke(chat_prompt)

json_response = parser.invoke(response)
print(json_response)

# 写法二
chain = chat_prompt_template | chat_model | parser
json_result = chain.invoke(input={"role":"人工智能专家","question":"人工智能用英文怎么说？问题用Q表示，答案用A表示，返回一个Json的数据。"})
print(json_result)

{'Q': '人工智能用英文怎么说？', 'A': 'Artificial Intelligence'}
{'Q': '人工智能用英文怎么说？', 'A': 'Artificial Intelligence'}


# 3、 XML解析器 XMLOutputParser

方法1：用户自己通过提示词指明返回XML格式

In [5]:
chat_model = ChatOpenAI(model = "gpt-4o-mini")

response = chat_model.invoke("请生成薛之谦专辑“天外来物”的简短歌曲记录，将歌曲附在<song></song>标签中")
print(response.content)

专辑《天外来物》是薛之谦于2017年发行的一张专辑，包含多首广受欢迎的歌曲。以下是几首歌曲的简短记录：

<song>
  <title>演员</title>
  <description>这首歌讲述了在感情中的角色扮演，表达了爱情中的无奈与心酸。</description>
</song>

<song>
  <title>换季</title>
  <description>通过换季的意象，展现出对逝去爱情的怀念与感伤，旋律动人。</description>
</song>

<song>
  <title>丑八怪</title>
  <description>歌曲传递了内心真实的自我认同与对外界审美压力的反思，鼓励人们做真实的自己。</description>
</song>

<song>
  <title>过客</title>
  <description>描述了人与人之间的匆匆过客关系，充满了淡淡的忧伤与思考。</description>
</song>

这些歌曲在情感表达上都非常细腻，展现了薛之谦独特的音乐风格和对生活的深刻理解。


方法2：使用get_format_instructions()

In [9]:
from langchain_core.output_parsers import XMLOutputParser

actor_query = "生成熊出没系列大电影的简短记录，使用中文回复"
# 定义XMLOutputParser对象
parser = XMLOutputParser()
# 生成提示词模版
prompt_template = PromptTemplate.from_template(
    template = "用户的问题：{query}\n 使用的格式：{format_instructions}"
)
template = prompt_template.partial(format_instructions = parser.get_format_instructions())
response = chat_model.invoke(template.invoke(input = {"query":actor_query}))
print(response.content)

```xml
<熊出没系列大电影>
   <电影>
      <标题>熊出没之夺宝熊兵</标题>
      <年份>2014</年份>
      <简介>讲述了熊大、熊二与光头强一起寻找宝藏的冒险故事。</简介>
      <角色>
         <熊大>主角，勇敢机智。</熊大>
         <熊二>熊大的好兄弟，搞笑可爱。</熊二>
         <光头强>反派角色，企图夺宝。</光头强>
      </角色>
   </电影>
   <电影>
      <标题>熊出没之雪岭熊风</标题>
      <年份>2015</年份>
      <简介>熊大和熊二在雪山中与光头强展开的滑雪大战。</简介>
      <角色>
         <熊大>主角，勇敢机智。</熊大>
         <熊二>熊大的好兄弟，搞笑可爱。</熊二>
         <光头强>反派角色，企图夺宝。</光头强>
      </角色>
   </电影>
   <电影>
      <标题>熊出没之奇幻空间</标题>
      <年份>2016</年份>
      <简介>熊大、熊二进入神秘空间，展开惊险的冒险之旅。</简介>
      <角色>
         <熊大>主角，勇敢机智。</熊大>
         <熊二>熊大的好兄弟，搞笑可爱。</熊二>
         <光头强>反派角色，企图实现自己的阴谋。</光头强>
      </角色>
   </电影>
   <电影>
      <标题>熊出没·变形记</标题>
      <年份>2017</年份>
      <简介>熊大、熊二变身成为各种角色，保护森林的故事。</简介>
      <角色>
         <熊大>主角，勇敢机智。</熊大>
         <熊二>熊大的好兄弟，搞笑可爱。</熊二>
         <光头强>反派角色，企图夺取森林资源。</光头强>
      </角色>
   </电影>
   <电影>
      <标题>熊出没·新年嘉年华</标题>
      <年份>2018</年份>
      <简介>在新年到来之际，熊大和熊二与朋友们一起迎接节日的欢庆。</简介>
      <角色>
         <熊大>主角，勇敢机智。<

XMLOutputParser 不会直接将模型的输出保持为原始XML字符串，而是会解析XML并转换成Python字典（或类似结构化的数据）。目的是为了方便程序后续处理数据，而不是单纯保留XML格式

In [10]:
from langchain_core.output_parsers import XMLOutputParser
parser = XMLOutputParser()
xml_result = parser.invoke(response)
print(xml_result)

{'熊出没系列大电影': [{'电影': [{'标题': '熊出没之夺宝熊兵'}, {'年份': '2014'}, {'简介': '讲述了熊大、熊二与光头强一起寻找宝藏的冒险故事。'}, {'角色': [{'熊大': '主角，勇敢机智。'}, {'熊二': '熊大的好兄弟，搞笑可爱。'}, {'光头强': '反派角色，企图夺宝。'}]}]}, {'电影': [{'标题': '熊出没之雪岭熊风'}, {'年份': '2015'}, {'简介': '熊大和熊二在雪山中与光头强展开的滑雪大战。'}, {'角色': [{'熊大': '主角，勇敢机智。'}, {'熊二': '熊大的好兄弟，搞笑可爱。'}, {'光头强': '反派角色，企图夺宝。'}]}]}, {'电影': [{'标题': '熊出没之奇幻空间'}, {'年份': '2016'}, {'简介': '熊大、熊二进入神秘空间，展开惊险的冒险之旅。'}, {'角色': [{'熊大': '主角，勇敢机智。'}, {'熊二': '熊大的好兄弟，搞笑可爱。'}, {'光头强': '反派角色，企图实现自己的阴谋。'}]}]}, {'电影': [{'标题': '熊出没·变形记'}, {'年份': '2017'}, {'简介': '熊大、熊二变身成为各种角色，保护森林的故事。'}, {'角色': [{'熊大': '主角，勇敢机智。'}, {'熊二': '熊大的好兄弟，搞笑可爱。'}, {'光头强': '反派角色，企图夺取森林资源。'}]}]}, {'电影': [{'标题': '熊出没·新年嘉年华'}, {'年份': '2018'}, {'简介': '在新年到来之际，熊大和熊二与朋友们一起迎接节日的欢庆。'}, {'角色': [{'熊大': '主角，勇敢机智。'}, {'熊二': '熊大的好兄弟，搞笑可爱。'}, {'光头强': '反派角色，试图破坏新年庆祝活动。'}]}]}]}


# 4、列表解析器CommaSeparatedListOutputParser(了解)

示例1：

In [11]:
from langchain_core.output_parsers import CommaSeparatedListOutputParser
output_parser = CommaSeparatedListOutputParser()
# 返回一些指令或模板，这些指令告诉系统如何解析或格式化输出数据
format_instructions = output_parser.get_format_instructions()
print(format_instructions)
messages = "大象,猩猩,狮子"
result = output_parser.parse(messages)
print(result)
print(type(result)) # list

Your response should be a list of comma separated values, eg: `foo, bar, baz` or `foo,bar,baz`
['大象', '猩猩', '狮子']
<class 'list'>


# 5、日期解析器DatetimeOutputParser(了解)

示例1：

In [19]:
from langchain_classic.output_parsers import DatetimeOutputParser
# from langchain_core.output_parsers import DatetimeOutputParser
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

chat_model = ChatOpenAI(model="gpt-4o-mini")
chat_prompt = ChatPromptTemplate.from_messages([
    ("system","{format_instructions}"),
    ("human", "{request}")
])
output_parser = DatetimeOutputParser()

# 方式1：
# model_request = chat_prompt.format_messages(
#     request="中华人民共和国是什么时候成立的",
#     format_instructions=output_parser.get_format_instructions()
# )
# response = chat_model.invoke(model_request)
# result = output_parser.invoke(response)
# print(result)
# print(type(result))

# 方式2：链式
chain = chat_prompt | chat_model | output_parser
resp = chain.invoke({"request":"中华人民共和国是什么时候成立的",
                     "format_instructions":output_parser.get_format_instructions()})
print(resp)
print(type(resp))

1949-10-01 00:00:00
<class 'datetime.datetime'>
